# 🍽️ Recipe Recommendation System
### Content-Based Filtering using TF-IDF + Cosine Similarity
This notebook builds a user-facing recipe recommender that takes:
- Ingredients the user has on hand
- Priority ingredients to heavily boost (e.g. the protein)
- Max prep time
- Max cook time
- Cuisine preference
- Minimum ingredient match threshold

And returns the top matching recipes ranked by relevance.

## Cell 1 — Install & Import Libraries

In [ ]:
!pip install pandas scikit-learn numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', None)
print('✅ Libraries loaded successfully')

## Cell 2 — Load & Preview the Dataset

In [ ]:
df = pd.read_csv('allrecipes_dataset.csv')
print(f'Dataset shape: {df.shape}')
print(f'Cuisines found: {sorted(df["cuisine"].unique())}')
df.head()

## Cell 3 — Clean & Preprocess the Data

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=['url'])
after = len(df)
print(f'🧹 Removed {before - after} duplicate recipes ({after} remaining)')

df = df.dropna(subset=['ingredients', 'title'])

df['prep_time_min']  = pd.to_numeric(df['prep_time_min'],  errors='coerce')
df['cook_time_min']  = pd.to_numeric(df['cook_time_min'],  errors='coerce')
df['total_time_min'] = pd.to_numeric(df['total_time_min'], errors='coerce')
df['rating']         = pd.to_numeric(df['rating'],         errors='coerce')

df['prep_time_min'].fillna(df['prep_time_min'].median(),   inplace=True)
df['cook_time_min'].fillna(df['cook_time_min'].median(),   inplace=True)
df['total_time_min'].fillna(df['total_time_min'].median(), inplace=True)
df['rating'].fillna(df['rating'].median(),                 inplace=True)

df['cuisine'] = df['cuisine'].str.strip().str.lower()

df['ingredients_clean'] = (
    df['ingredients']
    .str.lower()
    .str.replace(r'[^a-z\s|]', '', regex=True)
    .str.replace('|', ' ', regex=False)
)

df = df.reset_index(drop=True)
print(f'✅ Cleaned dataset: {len(df)} recipes across {df["cuisine"].nunique()} cuisines')

## Cell 4 — Build the TF-IDF Matrix

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(df['ingredients_clean'])
print(f'✅ TF-IDF matrix: {tfidf_matrix.shape[0]} recipes × {tfidf_matrix.shape[1]} ingredient terms')

## Cell 5 — Recommendation Function (Experiment 1 Baseline)

In [ ]:
def recommend_recipes(
    user_ingredients,
    max_prep_time=None,
    max_cook_time=None,
    cuisine=None,
    top_n=5,
    priority_ingredients=None,
    priority_boost=5.0,
    min_match=5.0,
    use_rating=True,       # ablation toggle: set False to remove rating
    use_prep_time=True,    # ablation toggle: set False to remove prep filter
    use_cook_time=True,    # ablation toggle: set False to remove cook filter
    use_priority=True      # ablation toggle: set False to remove priority boost
):
    """
    Recommend recipes based on user inputs.
    Ablation toggles allow Experiment 2 to remove one feature at a time.
    """
    filtered = df.copy()

    # Hard filters — only applied if the toggle is on
    if use_prep_time and max_prep_time is not None:
        filtered = filtered[filtered['prep_time_min'] <= max_prep_time]

    if use_cook_time and max_cook_time is not None:
        filtered = filtered[filtered['cook_time_min'] <= max_cook_time]

    if cuisine is not None:
        filtered = filtered[filtered['cuisine'] == cuisine.strip().lower()]

    if filtered.empty:
        print('⚠️  No recipes match your filters.')
        return None

    # Build boosted user vector
    all_ingredients = [i.strip().lower() for i in user_ingredients.split(',')]
    priority_list   = [p.strip().lower() for p in priority_ingredients.split(',')]\
                      if (priority_ingredients and use_priority) else []

    boosted_parts = []
    for ingredient in all_ingredients:
        if ingredient in priority_list:
            boosted_parts.extend([ingredient] * int(priority_boost))
        else:
            boosted_parts.append(ingredient)

    user_vector = tfidf.transform([' '.join(boosted_parts)])

    # Cosine similarity
    filtered_indices = filtered.index.tolist()
    filtered_tfidf   = tfidf_matrix[filtered_indices]
    similarities     = cosine_similarity(user_vector, filtered_tfidf).flatten()

    # Min match threshold
    similarity_pct  = similarities * 100
    above_threshold = similarity_pct >= min_match
    if not any(above_threshold):
        print(f'⚠️  No recipes scored above {min_match}% match.')
        return None

    filtered     = filtered.iloc[above_threshold]
    similarities = similarities[above_threshold]
    similarity_pct = similarity_pct[above_threshold]

    # Normalized rating — set to 0 if ablated
    if use_rating:
        max_r = df['rating'].max()
        min_r = df['rating'].min()
        norm_ratings = (filtered['rating'].values - min_r) / (max_r - min_r if max_r != min_r else 1)
    else:
        norm_ratings = np.zeros(len(filtered))

    # Weighted relevance score
    INGREDIENT_WEIGHT = 0.80
    RATING_WEIGHT     = 0.20
    relevance_scores  = INGREDIENT_WEIGHT * similarities + RATING_WEIGHT * norm_ratings

    results = filtered.copy()
    results['ingredient_match'] = similarity_pct.round(1)
    results['relevance_score']  = (relevance_scores * 100).round(1)

    results = results.sort_values('relevance_score', ascending=False).head(top_n)

    display_cols = ['title', 'cuisine', 'prep_time_min', 'cook_time_min',
                    'rating', 'ingredient_match', 'relevance_score', 'url']
    return results[display_cols].reset_index(drop=True)

print('✅ Recommender function ready!')

## Cell 6 — Try It Out!

In [ ]:
# ── ✏️  EDIT THESE INPUTS ─────────────────────────────────────────────────────
MY_INGREDIENTS       = 'chicken, garlic, lemon, olive oil, onion, salt, pepper'
PRIORITY_INGREDIENTS = 'chicken'
MAX_PREP_TIME        = 20
MAX_COOK_TIME        = 30
CUISINE              = None
HOW_MANY             = 10
MIN_MATCH            = 5.0

results = recommend_recipes(
    user_ingredients     = MY_INGREDIENTS,
    max_prep_time        = MAX_PREP_TIME,
    max_cook_time        = MAX_COOK_TIME,
    cuisine              = CUISINE,
    top_n                = HOW_MANY,
    priority_ingredients = PRIORITY_INGREDIENTS,
    min_match            = MIN_MATCH
)

if results is not None:
    print(f'🍽️  Top {HOW_MANY} recipes:')
    print(f'   Ingredients : {MY_INGREDIENTS}')
    print(f'   Priority    : {PRIORITY_INGREDIENTS}')
    print(f'   Max prep    : {MAX_PREP_TIME} min | Max cook: {MAX_COOK_TIME} min | Cuisine: {CUISINE or "Any"}')
    print()
    display(results)

## Cell 7 — Dataset Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cuisine_counts = df['cuisine'].value_counts()
axes[0].barh(cuisine_counts.index, cuisine_counts.values, color='steelblue')
axes[0].set_title('Recipes per Cuisine')
axes[0].set_xlabel('Count')

axes[1].hist(df['rating'].dropna(), bins=20, color='coral', edgecolor='white')
axes[1].set_title('Rating Distribution')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Count')

axes[2].hist(df['prep_time_min'].clip(upper=120), bins=20, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Prep Time Distribution (capped 120 min)')
axes[2].set_xlabel('Minutes')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()
print(f'Total recipes: {len(df)}')
print(df[['prep_time_min', 'cook_time_min', 'rating']].describe().round(1))

## Cell 8 — Evaluate: Precision Score (Baseline)

In [ ]:
def evaluate_precision(results, user_ingredients, priority_ingredients=None,
                       match_threshold=11.5, min_additional=2):
    """
    A recipe is truly relevant if:
      1. ingredient_match >= match_threshold  (Option A)
      2. contains the priority ingredient     (Option C rule 1)
      3. contains >= min_additional others    (Option C rule 2)
    """
    if results is None or results.empty:
        print('No results to evaluate.')
        return 0.0

    all_ingredients   = [i.strip().lower() for i in user_ingredients.split(',')]
    priority_list     = [p.strip().lower() for p in priority_ingredients.split(',')]\
                        if priority_ingredients else []
    other_ingredients = [i for i in all_ingredients if i not in priority_list]
    relevant_count    = 0

    print(f'\n📊 Evaluating {len(results)} recommendations:')
    print(f'   Threshold  : {match_threshold}% match score')
    print(f'   Must contain priority ingredient(s): {priority_list}')
    print(f'   Must contain at least {min_additional} other ingredients from your list')
    print()

    for _, row in results.iterrows():
        recipe_ingredients = df.loc[df['url'] == row['url'], 'ingredients_clean'].values[0]

        passes_threshold  = row['ingredient_match'] >= match_threshold
        contains_priority = all(p in recipe_ingredients for p in priority_list) if priority_list else True
        additional_matches = sum(1 for i in other_ingredients if i in recipe_ingredients)
        passes_additional  = additional_matches >= min_additional
        is_relevant        = passes_threshold and contains_priority and passes_additional

        status = '✅' if is_relevant else '❌'
        print(f'  {status} {row["title"][:45]:<45} | match: {row["ingredient_match"]}% | '
              f'priority: {"✓" if contains_priority else "✗"} | other matches: {additional_matches}')

        if is_relevant:
            relevant_count += 1

    precision = relevant_count / len(results)
    print(f'\n🎯 Precision: {precision:.2f} ({relevant_count}/{len(results)} recipes truly relevant)')
    return precision

# Run baseline evaluation
results = recommend_recipes(
    user_ingredients=MY_INGREDIENTS, max_prep_time=MAX_PREP_TIME,
    max_cook_time=MAX_COOK_TIME, cuisine=CUISINE, top_n=HOW_MANY,
    priority_ingredients=PRIORITY_INGREDIENTS, min_match=MIN_MATCH
)

BASELINE_PRECISION = evaluate_precision(
    results=results, user_ingredients=MY_INGREDIENTS,
    priority_ingredients=PRIORITY_INGREDIENTS,
    match_threshold=11.5, min_additional=2
)
print(f'\n📌 Baseline precision saved: {BASELINE_PRECISION:.2f}')

---
## Experiment 2 — Feature Ablation Study
We systematically remove one feature at a time and measure how precision changes.
This tells us which features actually contribute to recommendation quality.

Features tested:
- **No Rating** — rating weight set to zero, only ingredient match drives score
- **No Prep Time Filter** — prep time constraint is ignored
- **No Cook Time Filter** — cook time constraint is ignored  
- **No Priority Boost** — all ingredients treated equally, no boosting

In [ ]:
# ── Ablation configurations ───────────────────────────────────────────────────
# Each entry removes exactly one feature from the full model
ablation_configs = [
    {
        'name':         'Baseline (All Features)',
        'use_rating':   True,
        'use_prep':     True,
        'use_cook':     True,
        'use_priority': True,
    },
    {
        'name':         'No Rating',
        'use_rating':   False,
        'use_prep':     True,
        'use_cook':     True,
        'use_priority': True,
    },
    {
        'name':         'No Prep Time Filter',
        'use_rating':   True,
        'use_prep':     False,
        'use_cook':     True,
        'use_priority': True,
    },
    {
        'name':         'No Cook Time Filter',
        'use_rating':   True,
        'use_prep':     True,
        'use_cook':     False,
        'use_priority': True,
    },
    {
        'name':         'No Priority Boost',
        'use_rating':   True,
        'use_prep':     True,
        'use_cook':     True,
        'use_priority': False,
    },
]

ablation_results = []

for config in ablation_configs:
    print(f"\n{'='*55}")
    print(f"  Running: {config['name']}")
    print(f"{'='*55}")

    res = recommend_recipes(
        user_ingredients     = MY_INGREDIENTS,
        max_prep_time        = MAX_PREP_TIME,
        max_cook_time        = MAX_COOK_TIME,
        cuisine              = CUISINE,
        top_n                = HOW_MANY,
        priority_ingredients = PRIORITY_INGREDIENTS,
        min_match            = MIN_MATCH,
        use_rating           = config['use_rating'],
        use_prep_time        = config['use_prep'],
        use_cook_time        = config['use_cook'],
        use_priority         = config['use_priority'],
    )

    if res is not None:
        avg_match = res['ingredient_match'].mean()
        precision = evaluate_precision(
            results              = res,
            user_ingredients     = MY_INGREDIENTS,
            priority_ingredients = PRIORITY_INGREDIENTS,
            match_threshold      = 11.5,
            min_additional       = 2
        )
        ablation_results.append({
            'Configuration':      config['name'],
            'Precision':          round(precision, 2),
            'Avg Ingredient Match (%)': round(avg_match, 1),
            'Recipes Returned':   len(res)
        })
    else:
        ablation_results.append({
            'Configuration':      config['name'],
            'Precision':          0.0,
            'Avg Ingredient Match (%)': 0.0,
            'Recipes Returned':   0
        })

# ── Summary table ─────────────────────────────────────────────────────────────
ablation_df = pd.DataFrame(ablation_results)
print(f"\n{'='*55}")
print('📊 EXPERIMENT 2 — ABLATION STUDY SUMMARY')
print(f"{'='*55}")
display(ablation_df)

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if i == 0 else 'coral' for i in range(len(ablation_df))]
bars = ax.bar(ablation_df['Configuration'], ablation_df['Precision'], color=colors, edgecolor='white')

# Label each bar with its precision value
for bar, val in zip(bars, ablation_df['Precision']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Experiment 2 — Feature Ablation: Precision by Configuration', fontsize=13)
ax.set_ylabel('Precision')
ax.set_ylim(0, 1.1)
ax.axhline(y=BASELINE_PRECISION, color='steelblue', linestyle='--', alpha=0.5, label='Baseline')
ax.legend()
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('experiment2_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Chart saved as experiment2_ablation.png')

---
## Experiment 3 — Binary Classification
We reframe the recommendation problem as a binary classification task.

**Label definition:** A recipe is labeled **Recommended (1)** if:
- It matches at least 60% of the user's available ingredients AND
- Its total time is within the user's time constraints

Otherwise it is labeled **Not Recommended (0)**.

We train and compare two classifiers:
- **Logistic Regression** — a linear model that predicts probability of recommendation
- **Naive Bayes** — a probabilistic model that works well with text/ingredient data

Evaluation metrics: Precision, Recall, F1-Score

In [ ]:
# ── Step 1: Generate binary labels for every recipe ───────────────────────────
# A recipe is "recommended" if it matches >= 60% of user ingredients
# AND fits within the user's time constraints

def compute_match_score(recipe_ingredients, user_ingredients_list):
    """Compute what fraction of user ingredients appear in the recipe."""
    matches = sum(1 for ing in user_ingredients_list if ing in recipe_ingredients)
    return matches / len(user_ingredients_list) if user_ingredients_list else 0

user_ing_list  = [i.strip().lower() for i in MY_INGREDIENTS.split(',')]
max_total_time = (MAX_PREP_TIME or 9999) + (MAX_COOK_TIME or 9999)

# Compute match score for every recipe in the dataset
df['match_score'] = df['ingredients_clean'].apply(
    lambda x: compute_match_score(str(x), user_ing_list)
)

# Label: 1 if match >= 60% AND within total time budget
MATCH_THRESHOLD = 0.60
df['label'] = (
    (df['match_score'] >= MATCH_THRESHOLD) &
    (df['total_time_min'] <= max_total_time)
).astype(int)

print(f'Label distribution:')
print(df['label'].value_counts())
print(f'\nRecommended (1): {df["label"].sum()} recipes')
print(f'Not Recommended (0): {(df["label"]==0).sum()} recipes')

# Check we have enough positive examples to train
if df['label'].sum() < 5:
    print('\n⚠️  Very few positive examples — try lowering MATCH_THRESHOLD or relaxing time constraints')

In [ ]:
# ── Step 2: Prepare features and split data ───────────────────────────────────
# Features: TF-IDF ingredient vectors (same matrix built in Cell 4)
# Labels:   binary recommended/not recommended

X = tfidf_matrix          # TF-IDF ingredient features
y = df['label'].values    # binary labels

# 80% train, 20% test — stratified so both classes appear in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape[0]} recipes ({y_train.sum()} recommended)')
print(f'Test set : {X_test.shape[0]} recipes ({y_test.sum()} recommended)')

In [ ]:
# ── Step 3: Train and evaluate Logistic Regression ───────────────────────────
print('Training Logistic Regression...')
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

lr_precision = precision_score(y_test, lr_preds, zero_division=0)
lr_recall    = recall_score(y_test, lr_preds, zero_division=0)
lr_f1        = f1_score(y_test, lr_preds, zero_division=0)

print(f'\n📊 Logistic Regression Results:')
print(f'   Precision : {lr_precision:.2f}')
print(f'   Recall    : {lr_recall:.2f}')
print(f'   F1 Score  : {lr_f1:.2f}')
print(f'\nDetailed Report:')
print(classification_report(y_test, lr_preds, target_names=['Not Recommended', 'Recommended']))

In [ ]:
# ── Step 4: Train and evaluate Naive Bayes ────────────────────────────────────
# Multinomial Naive Bayes works well with TF-IDF features (text/ingredient data)
print('Training Naive Bayes...')
nb_model = MultinomialNB(alpha=1.0)   # alpha is the smoothing parameter
nb_model.fit(X_train, y_train)
nb_preds = nb_model.predict(X_test)

nb_precision = precision_score(y_test, nb_preds, zero_division=0)
nb_recall    = recall_score(y_test, nb_preds, zero_division=0)
nb_f1        = f1_score(y_test, nb_preds, zero_division=0)

print(f'\n📊 Naive Bayes Results:')
print(f'   Precision : {nb_precision:.2f}')
print(f'   Recall    : {nb_recall:.2f}')
print(f'   F1 Score  : {nb_f1:.2f}')
print(f'\nDetailed Report:')
print(classification_report(y_test, nb_preds, target_names=['Not Recommended', 'Recommended']))

In [ ]:
# ── Step 5: Compare classifiers visually ─────────────────────────────────────
metrics      = ['Precision', 'Recall', 'F1 Score']
lr_scores    = [lr_precision, lr_recall, lr_f1]
nb_scores    = [nb_precision, nb_recall, nb_f1]

x     = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, lr_scores, width, label='Logistic Regression', color='steelblue', edgecolor='white')
bars2 = ax.bar(x + width/2, nb_scores, width, label='Naive Bayes',         color='coral',     edgecolor='white')

# Label bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Experiment 3 — Logistic Regression vs Naive Bayes', fontsize=13)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
plt.tight_layout()
plt.savefig('experiment3_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Chart saved as experiment3_classification.png')

# ── Final comparison summary ───────────────────────────────────────────────────
summary_df = pd.DataFrame({
    'Model':     ['Logistic Regression', 'Naive Bayes'],
    'Precision': [round(lr_precision,2), round(nb_precision,2)],
    'Recall':    [round(lr_recall,2),    round(nb_recall,2)],
    'F1 Score':  [round(lr_f1,2),        round(nb_f1,2)],
})
print('\n📊 EXPERIMENT 3 — CLASSIFICATION SUMMARY')
display(summary_df)